# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [13]:


%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [14]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [15]:
import os
from glob import glob

PRICE_DATA_PATH = os.getenv("PRICE_DATA")
print(PRICE_DATA_PATH)



../../05_src/data/prices/


In [16]:
os.path.join(PRICE_DATA_PATH, "**/part.0.parquet")

'../../05_src/data/prices/**/part.0.parquet'

In [17]:
parquet_files = glob (os.path.join(PRICE_DATA_PATH, "**/part.0.parquet"),recursive= True)
dd.read_parquet(parquet_files)


,Date,Adj Close,Close,High,Low,Open,Volume,Year
npartitions=13078,,,,,,,,
,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32
,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...


In [18]:
dask_df = dd.read_parquet(parquet_files)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [19]:
dask_df = dask_df.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag=x['Close'].shift(1),
        Adj_Close_lag=x['Adj Close'].shift(1)
    )
)



/var/folders/vd/xp65dp3d05g14dvm__vskb7m0000gn/T/ipykernel_2911/3184130158.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dask_df = dask_df.groupby('Ticker', group_keys=False).apply(


In [20]:
dask_df['returns'] = (dask_df['Adj Close'] / dask_df['Adj_Close_lag']) - 1
#dask_df['returns']

In [21]:
dask_df['hi_lo_range'] = dask_df['High'] - dask_df['Low']
#dask_df['hi_lo_range']


In [22]:
dd_feat = dask_df

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [23]:
import pandas as pd
pd_feat = dd_feat.compute()

In [24]:
pd_feat['rolling_mean'] = pd_feat["returns"].rolling(10).mean()
pd_feat



Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag,Adj_Close_lag,returns,hi_lo_range,rolling_mean
Ticker,,,,,,,,,,,,,
DOV,2001-01-02 00:00:00+00:00,16.501926,25.891714,27.190489,25.556545,27.190489,725607.0,2001,NaN,NaN,NaN,1.633944,NaN
DOV,2001-01-03 00:00:00+00:00,17.730217,27.818930,27.986513,25.556545,26.143089,1635599.0,2001,25.891714,16.501926,0.074433,2.429968,NaN
DOV,2001-01-04 00:00:00+00:00,17.783621,27.902721,28.489265,27.399969,27.693241,904174.0,2001,27.818930,17.730217,0.003012,1.089296,NaN
DOV,2001-01-05 00:00:00+00:00,17.302984,27.148594,28.154097,26.562050,27.986513,1262651.0,2001,27.902721,17.783621,-0.027027,1.592047,NaN
DOV,2001-01-08 00:00:00+00:00,17.383095,27.274281,27.777033,26.897217,27.399969,1173740.0,2001,27.148594,17.302984,0.004630,0.879816,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
CTLT,2010-12-27 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN,NaN,NaN
CTLT,2010-12-28 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN,NaN,NaN
CTLT,2010-12-29 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN,NaN,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary to convert to pandas to calculate the moving average return. We could have done this in Dask. In fact, given the size of this dataset, it would probably be faster to have done it in Dask in the event that one is on a computer with less computing power (particularly less RAM) since Dask is better than Pandas at handling relatively large datasets. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.